Robust system path setup for project modules


In [3]:
# Cell 1: Robust System Path Setup for Project Modules
import os
import sys

notebook_path = os.getcwd() # This should be 'gaias_ark_mangroves/notebooks/'
project_root = os.path.abspath(os.path.join(notebook_path, os.pardir))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root added to sys.path: {project_root}")
print(f"'configs' folder exists at root: {os.path.isdir(os.path.join(project_root, 'configs'))}")
print(f"'regions.py' file exists: {os.path.isfile(os.path.join(project_root, 'configs', 'regions.py'))}")

Project root added to sys.path: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark
'configs' folder exists at root: True
'regions.py' file exists: True


 Import Core Libraries and GEE Initialization 

In [4]:
# Cell 2: Import Core Libraries and Initialize GEE
import ee # Google Earth Engine API
import pandas as pd # Data manipulation
import geopandas as gpd # Geospatial data manipulation (builds on pandas)
import folium # For interactive mapping in notebooks
from configs.regions import kenyan_coast_roi # Import your defined region from config

# Initialize GEE (essential for all GEE operations)
ee.Initialize(project='gaias-ark') 

print("All core libraries imported and GEE initialized.")

Region of Interest for Kenyan Coast defined.
All core libraries imported and GEE initialized.


Access GEDI Level 2A Canopy Height Data


In [5]:
# Cell 3: Access GEDI L4A Biomass Data (Expanded Date Range)
print("--- Accessing GEDI L4A Monthly Biomass Data ---")

gedi_collection_id = "LARSE/GEDI/GEDI04_A_002_MONTHLY"

gedi_data = ee.ImageCollection(gedi_collection_id)

# Filter by a much wider date range to capture more data (GEDI started April 2019)
start_date = '2019-04-01' # GEDI mission start
end_date = '2023-12-31'   # Up to a recent full year, or even '2025-10-01' for current data
gedi_filtered = gedi_data.filterDate(start_date, end_date)

gedi_in_roi_collection = gedi_filtered.filterBounds(kenyan_coast_roi)

num_gedi_images = gedi_in_roi_collection.size().getInfo()
print(f"GEDI L4A AGBD data accessed for {start_date} to {end_date}.")
print(f"Number of GEDI L4A monthly images in ROI: {num_gedi_images}")

if num_gedi_images == 0:
    print("CRITICAL WARNING: Still no GEDI L4A data found. Consider re-evaluating ROI or dataset.")
    # Create a dummy empty image. This will display as a blank map for this layer.
    gedi_agbd_image = ee.Image(0).rename('agbd').clip(kenyan_coast_roi)
else:
    # Take the median AGBD over the filtered collection of monthly images
    gedi_agbd_image = gedi_in_roi_collection.median().select('agbd').clip(kenyan_coast_roi)

--- Accessing GEDI L4A Monthly Biomass Data ---
GEDI L4A AGBD data accessed for 2019-04-01 to 2023-12-31.
Number of GEDI L4A monthly images in ROI: 48


Visualize GEDI Data (Points) on Folium Map


In [6]:
# Cell 4: Visualize GEDI L4A AGBD Raster on Folium Map
print("--- Visualizing GEDI L4A AGBD Raster ---")

# Define visualization parameters for AGBD (biomass density)
# AGBD values are in Mg/ha (megagrams per hectare), which is tonnes/hectare.
# Adjust min/max based on typical mangrove biomass values (e.g., 0 to 500 Mg/ha for mature mangroves).
agbd_vis_params = {
    'min': 0,
    'max': 300, # Max biomass for dense mangroves, adjust as needed for your region
    'palette': ['lightgreen', 'darkgreen', 'darkred'] # Example: low biomass light green, high biomass dark red
}

centroid_coords = kenyan_coast_roi.centroid().getInfo()['coordinates']
center_lat, center_lon = centroid_coords[1], centroid_coords[0]

m_gedi_agbd = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='OpenStreetMap')

roi_geojson = kenyan_coast_roi.getInfo()
folium.GeoJson(
    roi_geojson,
    name='Kenyan Coastal ROI',
    style_function=lambda x: {'fillColor': '#f08080', 'color': 'red', 'weight': 3, 'fillOpacity': 0.1}
).add_to(m_gedi_agbd)

# Add the GEDI AGBD raster layer
map_id_dict_agbd = gedi_agbd_image.getMapId(agbd_vis_params)
folium.TileLayer(
    tiles=map_id_dict_agbd['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='GEDI L4A AGBD (Mg/ha)'
).add_to(m_gedi_agbd)

folium.LayerControl().add_to(m_gedi_agbd)
m_gedi_agbd

--- Visualizing GEDI L4A AGBD Raster ---
